# Chapter 11 (Part 2): RAG Pipeline & LLM Analyst
**Financial AI in Practice** | Manning Publications

This notebook covers:
- Batch embedding with OpenAI API
- Vector database storage with ChromaDB
- Two-step retrieval strategy (Trend Scan + Deep Dive)
- Structured prompt engineering for quantifiable signal extraction
- Deploying the "LLM Analyst" for Policy Tone, Supply Risk, and Novelty scores

## 11.3.3 Embedding the unified dataset
### Listing 11.7 — Batch embedding with the OpenAI API

In [ ]:
from openai import OpenAI
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import os

df = pd.read_parquet('news_ready_for_embedding.parquet')

# Keys come from the repo-root .env (appendix A): copy .env.example to .env and fill it in
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set: copy .env.example to .env in the repo root and add your key.")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("="*80)
print("STEP 2: TEXT EMBEDDING")
print("="*80)
print(f"Total texts: {len(df)}")

texts = df['text_for_embedding'].tolist()
embeddings = []

batch_size = 100
for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
    batch = texts[i:i+batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend([item.embedding for item in response.data])

df['embedding'] = embeddings
df.to_parquet('news_embedded.parquet')
print(f"\nSaved: news_embedded.parquet")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")
    print(f"Sample embedding (first 5 dims): {embeddings[0][:5]}")
else:
    print("No embeddings were generated (empty input).")

### Listing 11.8 — Initializing ChromaDB and storing the embedded news

In [ ]:
import chromadb

df = pd.read_parquet('news_embedded.parquet')

print("=" * 80)
print("STEP 3: VECTOR DATABASE STORAGE")
print("=" * 80)

chroma_client = chromadb.PersistentClient(path="./chroma_db")
try:
    chroma_client.delete_collection("news")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection("news")
ids = [str(i) for i in range(len(df))]
collection.add(
    ids=ids,
    embeddings=df['embedding'].tolist(),
    documents=df['text_for_embedding'].tolist(),
    metadatas=df['metadata'].tolist()
)

print(f"Stored {len(df)} documents")
print(f"Collection name: {collection.name}")
print(f"Total count: {collection.count()}")

### Listing 11.9 — Verifying the RAG pipeline with a test query

In [ ]:
print("=" * 80)
print("VERIFICATION TEST")
print("=" * 80)

query = "uranium supply disruption"
query_emb = client.embeddings.create(
    model="text-embedding-3-small", input=[query]
).data[0].embedding

results = collection.query(query_embeddings=[query_emb], n_results=5)

print(f"\nQuery: '{query}'")
print(f"Retrieved: {len(results['ids'][0])} documents\n")

for i in range(len(results['ids'][0])):
    meta = results['metadatas'][0][i]
    doc = results['documents'][0][i]
    score = 1 - results['distances'][0][i]
    print(f"[{i+1}] Similarity: {score:.3f}")
    print(f"    Title: {meta['title']}")
    print(f"    Source: {meta['source']} ({meta['data_source']})")
    print(f"    Date: {meta['published_dt'][:10]}")
    print(f"    Text: {doc[:100]}...\n")

## 11.4 Building the LLM Analyst
### Listing 11.10 — Implementing the date-filtered retrieval strategy

In [ ]:
from datetime import datetime, timedelta

# Simulation: Analysis as of September 30, 2025
target_date = datetime(2025, 9, 30)
query = "Kazatomprom uranium production supply"

query_emb = client.embeddings.create(
    model="text-embedding-3-small", input=[query]
).data[0].embedding

# Step 1: Trend Scan (Last 30 days) - headlines for context
start_30d = int((target_date - timedelta(days=30)).timestamp())
end = int(target_date.timestamp())

results_trend = collection.query(
    query_embeddings=[query_emb], n_results=50,
    where={"$and": [{"published_ts": {"$gte": start_30d}}, {"published_ts": {"$lte": end}}]}
)
print(f"Trend Scan: {len(results_trend['ids'][0])} articles (30 days)")

# Step 2: Deep Dive (Last 3 days) - content-rich for analysis
start_3d = int((target_date - timedelta(days=3)).timestamp())

results_recent = collection.query(
    query_embeddings=[query_emb], n_results=10,
    where={"$and": [
        {"published_ts": {"$gte": start_3d}},
        {"published_ts": {"$lte": end}},
        {"has_content": True}
    ]}
)
print(f"Deep Dive: {len(results_recent['ids'][0])} content-rich articles (3 days)\n")

print("Recent articles:")
for i in range(min(5, len(results_recent['ids'][0]))):
    meta = results_recent['metadatas'][0][i]
    print(f"  {meta['published_dt'][:10]} | {meta['title']}")

### Listing 11.11 — Building the two-step context and structured prompt

In [ ]:
# Step 1: Build trend context from 30-day headlines
trend_headlines = []
for i in range(min(30, len(results_trend['ids'][0]))):
    meta = results_trend['metadatas'][0][i]
    trend_headlines.append(f"- {meta['published_dt'][:10]}: {meta['title']}")

trend_context = "\n".join(trend_headlines)

# Step 2: Build detailed context from 3-day content-rich articles
detail_docs = []
for i in range(len(results_recent['ids'][0])):
    meta = results_recent['metadatas'][0][i]
    doc = results_recent['documents'][0][i]
    detail_docs.append(f"[{meta['published_dt'][:10]} | {meta['source']}]\n{doc}\n")

detail_context = "\n".join(detail_docs)

# Structured prompt with both contexts
prompt = f"""You are a quantitative analyst analyzing uranium sector news.
Based ONLY on the context below, provide a structured assessment.

TREND CONTEXT (30-day headlines for momentum):
{trend_context}

DETAILED ANALYSIS (last 3 days with full content):
{detail_context}

First, review the 30-day headlines to understand the dominant narrative trends.
Then, analyze the detailed recent articles to extract specific signals.

OUTPUT (JSON only):
{{
  "sentiment_score": <-100 to +100, overall market sentiment>,
  "supply_risk": <0 to 10, likelihood of supply disruption>,
  "policy_tone": <-10 to +10, regulatory environment>,
  "conviction": <0 to 10, signal strength>,
  "key_events": ["event1", "event2"],
  "reasoning": "brief explanation"
}}
"""

# Call LLM
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}]
)

print("="*80)
print("LLM SIGNAL EXTRACTION")
print("="*80)
print(response.choices[0].message.content)